# Notebook 13: Investment Education Agent

## What This Notebook Does
Provides **personalised investment education** grounded in the investor's
actual consolidated portfolio produced by Notebook 12.

Every explanation follows a 4-step framework:
1. **Simple Definition** — one sentence, no jargon
2. **Real-World Analogy** — non-financial comparison that makes it stick
3. **Portfolio Application** — references the investor's real holdings
4. **Practical Takeaway** — one concrete consideration to act on

## Key Design Change from Previous Version
| | Before (hardcoded) | Now (dynamic) |
|--|-------------------|---------------|
| Portfolio source | Ticker list hard-coded in prompt | Loaded from `consolidated_portfolio.json` (NB12 output) |
| Holdings context | Static, never updated | Rebuilt from file every run |
| Works without NB12? | Yes (used hardcoded data) | Yes (graceful fallback with clear warning) |
| Portfolio value | Not available | Included when NB12 computed it |

## Architecture
```
consolidated_portfolio.json   (output of Notebook 12)
          │
          ▼
  load_portfolio_context()    ← reads NB12 JSON, builds dynamic system prompt
          │
          ▼
  Education Agent Graph
  ┌─────────────────────┐
  │  assistant ◄────────┤
  │      │              │
  │  tools?──► ToolNode─┘
  │      │
  │     END
  └─────────────────────┘
          │
          ▼
   Multi-turn education chat
   (portfolio context injected into every turn)
```

## Tools
| Tool | Purpose |
|------|---------|
| `search_web` | Current prices, news, and fund data for holdings |

## Three-Agent System
| Notebook | Role | Input | Output |
|----------|------|-------|--------|
| **11** | Profile investor, build portfolio | User conversation | `portfolio.json` |
| **12** | Consolidate & analyse | Portfolio folder | `consolidated_portfolio.json` |
| **13** (This) | Investment education | `consolidated_portfolio.json` | Education chat |

## How to Use This Notebook

### Prerequisites
1. **Run Notebook 12 first** so `consolidated_portfolio.json` exists.
   If you skip this step the notebook will warn you and continue with
   a generic (non-personalised) mode.
2. A `.env` file at `../.env` with:
   ```
   LLM_PROVIDER=openai
   LLM_MODEL=gpt-4.1-mini
   OPENAI_API_KEY=your-key
   SERPER_API_KEY=your-key
   ```

### Configuration (one variable)
```python
CONSOLIDATED_PORTFOLIO_FILE = "../data/outputs/consolidated_portfolio.json"
```
Change this path if you saved NB12's output somewhere else.

### Running
Use **Kernel > Restart & Run All** to:
1. Load your portfolio from NB12
2. Build a personalised system prompt containing all your holdings
3. Run 6 education demo queries that reference your real tickers
4. Open the interactive chat for free-form questions

### What to Ask
- `"What is beta and how does it affect my portfolio?"`
- `"Explain ETFs vs mutual funds using my holdings"`
- `"How does inflation impact my bond allocations?"`
- `"What is the Sharpe ratio of my top holdings?"`
- `"Explain tax-loss harvesting with examples from my portfolio"`

## Installation

In [ ]:
%pip install langchain langchain-openai langchain-community langgraph \
             python-dotenv pydantic google-search-results --quiet

print("Packages installed")

## Imports

In [1]:
import json
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from ai_course_utils import load_llm_from_env, display_config

# Load environment
load_dotenv()
load_dotenv('../.env')

print("Imports successful")

Imports successful


## Configuration

**Only one variable to set** — the path to Notebook 12's output file.
Everything else (tickers, allocations, dollar values) is read from that file.

In [2]:
# ── USER-DEFINED SETTINGS ─────────────────────────────────────────────────────

# Path to the JSON file saved by Notebook 12.
# Change this if you saved NB12's output in a different location.
CONSOLIDATED_PORTFOLIO_FILE = "../data/outputs/consolidated_portfolio.json"

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DIVIDER = "=" * 70

display_config()
print(f"\nPortfolio source : {os.path.abspath(CONSOLIDATED_PORTFOLIO_FILE)}")

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')

Portfolio source : /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/data/outputs/consolidated_portfolio.json


## Load Portfolio from Notebook 12

This cell reads the consolidated portfolio produced by NB12 and builds:
- A **human-readable holdings block** injected into the system prompt
- A **metadata summary** (number of holdings, total AUM, source files)

If the file is not found a clear warning is shown and a generic
(non-personalised) mode is activated.

In [3]:
def load_portfolio_context(filepath: str) -> dict:
    """
    Load consolidated_portfolio.json produced by Notebook 12.

    Returns a dict with:
        holdings        : list of holding dicts
        context_block   : formatted string for system prompt injection
        metadata        : summary info (n holdings, AUM, source files)
        personalised    : True if a real portfolio was loaded
    """
    path = Path(filepath)

    # ── File not found — graceful fallback ─────────────────────────────────
    if not path.exists():
        print(f"⚠  Portfolio file not found: {path.resolve()}")
        print("   The agent will run in GENERIC mode (no personalised holdings).")
        print("   Run Notebook 12 first to enable portfolio-personalised education.")
        return {
            "holdings":      [],
            "context_block": "No portfolio loaded. Provide general investment education.",
            "metadata":      {"source": "none", "n_holdings": 0, "total_value": None},
            "personalised":  False,
        }

    # ── Load and parse ─────────────────────────────────────────────────────
    with open(path) as f:
        data = json.load(f)

    # Support both NB12 output schema and plain NB11 portfolio.json
    # NB12 schema: {source_folder, source_files, total_holdings, holdings: [...]}
    # NB11 schema: {name, description, holdings: [...]}
    holdings = data.get("holdings", [])
    if not holdings:
        print(f"⚠  Portfolio file found but contains no holdings: {path.name}")
        return {
            "holdings":      [],
            "context_block": "Portfolio file was empty. Provide general investment education.",
            "metadata":      {"source": path.name, "n_holdings": 0, "total_value": None},
            "personalised":  False,
        }

    # ── Build the context block for system prompt injection ─────────────────
    lines = [
        "INVESTOR'S CONSOLIDATED PORTFOLIO",
        "-" * 52,
    ]

    total_value = data.get("total_value_usd")
    if total_value:
        lines.append(f"  Total AUM : ${total_value:,.0f}")

    source_files = data.get("source_files", [path.name])
    lines.append(f"  Sources   : {', '.join(source_files)}")
    lines.append(f"  Holdings  : {len(holdings)}")
    lines.append("-" * 52)

    for h in holdings:
        ticker    = h.get("ticker", "?")
        name      = h.get("company_name", ticker)
        inv_type  = h.get("investment_type", "Unknown")
        alloc_pct = h.get("allocation_pct", 0)
        value_usd = h.get("value_usd")

        val_str = f"  (${value_usd:>10,.0f})" if value_usd else ""
        lines.append(
            f"  {ticker:<8} | {inv_type:<14} | {alloc_pct:6.2f}%{val_str}  — {name}"
        )

    lines.append("-" * 52)
    context_block = "\n".join(lines)

    metadata = {
        "source":      path.name,
        "n_holdings":  len(holdings),
        "total_value": total_value,
        "source_files": source_files,
    }

    return {
        "holdings":      holdings,
        "context_block": context_block,
        "metadata":      metadata,
        "personalised":  True,
    }


# ── Run the loader ─────────────────────────────────────────────────────────────
portfolio_data = load_portfolio_context(CONSOLIDATED_PORTFOLIO_FILE)

print(DIVIDER)
if portfolio_data["personalised"]:
    print("PORTFOLIO LOADED SUCCESSFULLY")
    print(DIVIDER)
    print(portfolio_data["context_block"])
else:
    print("RUNNING IN GENERIC MODE — no portfolio file found")
print(DIVIDER)

⚠  Portfolio file not found: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/data/outputs/consolidated_portfolio.json
   The agent will run in GENERIC mode (no personalised holdings).
   Run Notebook 12 first to enable portfolio-personalised education.
RUNNING IN GENERIC MODE — no portfolio file found


## Build Dynamic System Prompt

The system prompt is constructed at runtime — the investor's actual
holdings are embedded directly so every LLM response can reference
their real tickers, allocations, and dollar values.

In [4]:
def build_system_prompt(portfolio_ctx: dict) -> str:
    """
    Build the education agent system prompt with the portfolio
    context block embedded.

    Called once after loading the portfolio, then injected into
    every conversation turn.
    """
    context_block = portfolio_ctx["context_block"]
    personalised  = portfolio_ctx["personalised"]

    personalisation_note = (
        "Always relate your explanations to the investor's specific holdings "
        "listed above. Reference real tickers and allocations when illustrating "
        "any concept."
        if personalised else
        "No portfolio is loaded. Provide general investment education with "
        "illustrative examples. Encourage the user to run Notebook 12 first "
        "for personalised explanations."
    )

    return f"""\
You are an expert investment educator helping an investor understand their
portfolio and financial concepts.

{context_block}

## Your Teaching Framework
For EVERY concept or term you explain, use this 4-step structure:

1. **Simple Definition** — one sentence, no jargon, no unexplained acronyms.
2. **Real-World Analogy** — a non-financial comparison that makes the concept
   immediately intuitive.
3. **Portfolio Application** — show exactly how this applies to the investor's
   own holdings (reference specific tickers and allocation percentages).
4. **Practical Takeaway** — one concrete action or consideration the investor
   can think about based on this concept.

## Personalisation Rule
{personalisation_note}

## Topics You Can Cover
- Financial terms: Beta, Alpha, Sharpe Ratio, P/E, Duration, Yield, Expense Ratio
- Asset allocation: Growth vs income, Modern Portfolio Theory, correlation
- ETF vs Mutual Fund: cost, tax efficiency, liquidity, trading differences
- Macroeconomic impact: interest rates, inflation, sector rotation
- Tax-efficient investing: Roth vs IRA, tax-loss harvesting, asset location
- Risk metrics: volatility, max drawdown, Sharpe ratio, concentration risk
- Portfolio theory: diversification, efficient frontier, rebalancing

## Web Search
Use the search_web tool when you need:
- Current prices or NAV for any holding
- Recent performance data or fund fact sheets
- Latest news that affects a holding the user is asking about
- Up-to-date expense ratios, yield, or fund composition

## Style Guidelines
- Be encouraging and conversational — this investor is building confidence.
- Never overwhelm with multiple questions. Ask at most one clarifying question.
- Keep responses focused: use the 4-step framework then stop.
- Always end with: "This is for educational purposes only, not financial advice."
"""


SYSTEM_PROMPT = build_system_prompt(portfolio_data)

print("System prompt built")
print(f"  Length      : {len(SYSTEM_PROMPT):,} characters")
print(f"  Personalised: {portfolio_data['personalised']}")
print(f"  Holdings    : {portfolio_data['metadata']['n_holdings']}")

System prompt built
  Length      : 2,029 characters
  Personalised: False
  Holdings    : 0


## Define Tools

One tool: **`search_web`** — lets the agent look up current data
for any holding in the portfolio (price, performance, news).

In [5]:
@tool
def search_web(query: str) -> str:
    """Search the web for current investment data — prices, fund facts,
    performance, expense ratios, or financial news for any holding."""
    try:
        search = GoogleSerperAPIWrapper()
        return search.run(query)
    except Exception as e:
        return f"Search error: {str(e)}"


tools = [search_web]
print(f"Tools ready: {[t.name for t in tools]}")

Tools ready: ['search_web']


## Build the Education Agent Graph

Same tool-calling loop pattern as Notebook 11:

```
assistant ◄──────────────┐
    │                    │
 tools? ──► ToolNode ────┘
    │
   END
```

The **system prompt is rebuilt each run** from the current portfolio file,
so if NB12 updates `consolidated_portfolio.json` the education agent
automatically picks up the latest holdings on next run.

In [6]:
# Initialise LLM
llm            = load_llm_from_env()
llm_with_tools = llm.bind_tools(tools)
memory         = MemorySaver()


def assistant(state: MessagesState):
    """Call the LLM with the dynamic system prompt + conversation history.

    SYSTEM_PROMPT is built at notebook load time from the actual portfolio
    file — no hardcoded tickers anywhere in this function.
    """
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}


def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """Route to tools if the LLM made tool calls, otherwise end."""
    return "tools" if state["messages"][-1].tool_calls else "__end__"


# Build graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("assistant", assistant)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "assistant")
graph_builder.add_conditional_edges("assistant", should_continue, ["tools", "__end__"])
graph_builder.add_edge("tools", "assistant")

graph = graph_builder.compile(checkpointer=memory)
print("Education Agent compiled")


def chat(user_message: str, session_id: str = "education") -> str:
    """Send a message to the education agent and return its response.

    Args:
        user_message : The investor's question.
        session_id   : Conversation thread — use different IDs to keep
                       parallel sessions independent.

    Returns:
        The agent's response string.
    """
    config   = {"configurable": {"thread_id": session_id}}
    response = graph.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config
    )
    return response["messages"][-1].content


print("chat() function ready")

🤖 Loading LLM: openai / gpt-4.1-mini
Education Agent compiled
chat() function ready


## Demo Queries

Six pre-built questions that showcase portfolio-personalised explanations.
Each runs in its own `session_id` so conversation history does not bleed
between demos.

| Demo | Concept | 4-Step Framework Focus |
|------|---------|------------------------|
| 1 | Beta | Risk relative to market |
| 2 | ETF vs Mutual Fund | Structure & cost differences |
| 3 | Inflation impact | Effect on bonds and equities |
| 4 | Diversification | Correlation across holdings |
| 5 | Sharpe Ratio | Risk-adjusted return |
| 6 | Tax-loss harvesting | Tax efficiency strategy |

In [11]:
print(DIVIDER)
print("DEMO 1: Sharp Ratio")
print(DIVIDER)
response = chat(
    "“What is a Sharpe ratio?”",
    session_id="demo1",
)
print(response)

DEMO 1: Sharp Ratio
Let's explore the Sharpe Ratio using our 4-step framework:

1. **Simple Definition**  
The Sharpe Ratio measures how much extra return you get from an investment compared to taking no risk, adjusted for how much the investment's value goes up and down.

2. **Real-World Analogy**  
Imagine you're choosing between two runners: one runs faster but stumbles a lot, the other runs a bit slower but steadily. The Sharpe Ratio tells you which runner is giving you the best balance of speed (return) and steadiness (risk).

3. **Portfolio Application**  
If you had a fund that returned 8% per year but its value bounced around a lot, and another fund that returned 6% but was much steadier, the Sharpe Ratio would help you see which fund offered better risk-adjusted returns, even if its raw return was lower.

4. **Practical Takeaway**  
When comparing investments or funds, look at their Sharpe Ratios to evaluate which ones offer better returns for the risk you're taking. Higher Sh

In [12]:
print(DIVIDER)
print("DEMO 2: Explain diversification")
print(DIVIDER)
response = chat(
    "Explain diversification and how does it affect the risk in my portfolio?",
    session_id="demo1",
)
print(response)

DEMO 2: Explain diversification
Happy to explain diversification with the 4-step framework!

1. **Simple Definition**  
Diversification means spreading your investments across different types of assets or sectors to reduce the chance that one loss will greatly hurt your whole portfolio.

2. **Real-World Analogy**  
Think of diversification like not putting all your eggs in one basket; if you drop one basket, you still have eggs safe in other baskets.

3. **Portfolio Application**  
If you owned stocks in just one industry, say technology, your portfolio could be very affected if that industry faces problems. But if you spread your money across technology, healthcare, bonds, and international stocks, losses in one area might be balanced by gains or stability in others, reducing your overall risk.

4. **Practical Takeaway**  
Aim to diversify your investments across different asset classes, sectors, and regions to help protect your portfolio from big drops. Consider checking your current

In [7]:
print(DIVIDER)
print("DEMO 1: Beta")
print(DIVIDER)
response = chat(
    "What is beta and how does it affect the risk in my portfolio?",
    session_id="demo1",
)
print(response)

DEMO 1: Beta
Certainly! Let's break down the concept of beta using the 4-step framework:

1. **Simple Definition**  
Beta is a number that measures how much a stock or investment tends to move up or down compared to the overall market.

2. **Real-World Analogy**  
Think of beta like a car's responsiveness to the gas pedal. If the market is the gas pedal, a beta greater than 1 means your investment speeds up faster than the market when it moves, while a beta less than 1 means it moves slower and is less reactive.

3. **Portfolio Application**  
If you had, for example, 50% in a technology ETF that typically has a beta around 1.2 and 50% in a stable utility stock with beta about 0.6, your portfolio's overall beta would be a blend of these. This means your portfolio would be somewhat more volatile than the market because the technology part amplifies market moves, while the utility part dampens them.

4. **Practical Takeaway**  
Consider checking the beta of your investments to understand

In [8]:
print(DIVIDER)
print("DEMO 2: ETF vs Mutual Fund")
print(DIVIDER)
response = chat(
    "Can you explain the difference between ETFs and mutual funds "
    "using examples from my portfolio?",
    session_id="demo2",
)
print(response)

DEMO 2: ETF vs Mutual Fund
I don’t see your portfolio loaded yet, so I can’t reference your specific holdings. If you’d like personalized examples, you can run Notebook 12 first and then share your portfolio here.

For now, I’ll explain the difference between ETFs and mutual funds in general terms using illustrative examples:

1. Simple Definition:
- ETFs (Exchange-Traded Funds) are investment funds traded on stock exchanges, like individual stocks. Mutual funds pool money from many investors to buy a diversified portfolio but are bought and sold only at the end of the trading day.

2. Real-World Analogy:
- Think of an ETF like a bakery that you can visit anytime during the day to buy a loaf of bread at the current price. A mutual fund is like placing a special bread order that’s made fresh once a day at a set price.

3. Portfolio Application:
- For example, suppose you owned an ETF like SPY (which tracks the S&P 500) and a mutual fund like Vanguard’s VFIAX (which also tracks the S&P 5

In [9]:
print(DIVIDER)
print("DEMO 3: Inflation Impact")
print(DIVIDER)
response = chat(
    "How does rising inflation affect my bond and equity holdings?",
    session_id="demo3",
)
print(response)

DEMO 3: Inflation Impact
Great question! Let's break down how rising inflation can impact both bonds and equities using our 4-step teaching framework:

1. Simple Definition:
- Inflation means the general rise in prices of goods and services over time.
- Rising inflation can decrease the value of bonds and impact stock prices because it affects costs, interest rates, and purchasing power.

2. Real-World Analogy:
Imagine you have a fixed allowance each week (like a bond’s fixed interest payment) and the prices at the grocery store keep going up (inflation). With higher prices, your allowance buys less than before — that’s what happens to bond investors when inflation rises. For stocks, it’s like running a store with rising costs and uncertain customer demand — profits may get squeezed, causing the store's value to go up or down unpredictably.

3. Portfolio Application:
- Bonds: Rising inflation means bonds with fixed interest payments are less attractive since the payments lose purchasin

In [ ]:
print(DIVIDER)
print("DEMO 4: Diversification")
print(DIVIDER)
response = chat(
    "How well diversified is my portfolio and what does correlation "
    "mean in this context?",
    session_id="demo4",
)
print(response)

In [10]:
print(DIVIDER)
print("DEMO 5: Sharpe Ratio")
print(DIVIDER)
response = chat(
    "What is the Sharpe Ratio and how should I think about it "
    "when evaluating my holdings?",
    session_id="demo5",
)
print(response)

DEMO 5: Sharpe Ratio
Let's break down the Sharpe Ratio in a simple and useful way:

1. **Simple Definition:**  
The Sharpe Ratio measures how much extra return you get from an investment compared to a risk-free option, adjusted for the amount of risk you take.

2. **Real-World Analogy:**  
Think of it like getting paid for doing a risky job: The Sharpe Ratio shows how much money you earn (reward) for each unit of risk (danger or effort) you take on in that job. A higher ratio means you’re getting more reward for the risk you’re accepting.

3. **Portfolio Application:**  
If you had two funds, Fund A with a Sharpe Ratio of 1.5 and Fund B with 0.5, this suggests Fund A is delivering a better balance of return versus risk. So if you had a holding like an S&P 500 ETF and a high-volatility tech ETF, the Sharpe Ratio helps you compare which is more efficient at generating returns for the risk involved.

4. **Practical Takeaway:**  
Look for investments with higher Sharpe Ratios when building

In [ ]:
print(DIVIDER)
print("DEMO 6: Tax-Loss Harvesting")
print(DIVIDER)
response = chat(
    "What is tax-loss harvesting and how could I apply it to "
    "my current portfolio?",
    session_id="demo6",
)
print(response)

## Portfolio Context Inspection

Print the exact context block that is injected into every conversation turn.
This confirms the agent is using your real holdings, not hardcoded data.

In [ ]:
print(DIVIDER)
print("PORTFOLIO CONTEXT INJECTED INTO EVERY PROMPT")
print(DIVIDER)
print(portfolio_data["context_block"])
print()
print(f"Personalised mode : {portfolio_data['personalised']}")
print(f"Source file       : {portfolio_data['metadata']['source']}")
print(f"Number of holdings: {portfolio_data['metadata']['n_holdings']}")
if portfolio_data["metadata"].get("total_value"):
    print(f"Total AUM         : ${portfolio_data['metadata']['total_value']:,.0f}")
print(DIVIDER)

## Reload Portfolio (optional)

If you re-run Notebook 12 and want to pick up an updated
`consolidated_portfolio.json` **without restarting the whole notebook**,
run this cell. It reloads the file, rebuilds the system prompt, and
recompiles the agent graph.

In [ ]:
# ── Run this cell to hot-reload the portfolio from NB12 ───────────────────────
portfolio_data = load_portfolio_context(CONSOLIDATED_PORTFOLIO_FILE)
SYSTEM_PROMPT  = build_system_prompt(portfolio_data)

# Recompile graph so the updated prompt takes effect immediately
graph = graph_builder.compile(checkpointer=MemorySaver())

print(DIVIDER)
print("Portfolio reloaded and agent recompiled")
print(DIVIDER)
print(portfolio_data["context_block"])
print(DIVIDER)

## Interactive Education Chat

Free-form conversation. The agent knows your exact holdings and can
search the web for current data on any of them.

**Conversation is stateful** within this session — you can ask follow-up
questions and the agent remembers what was discussed.

Type `quit`, `exit`, or `q` to stop.

In [ ]:
print(DIVIDER)
print("INVESTMENT EDUCATION AGENT — Interactive Mode")
print(DIVIDER)
if portfolio_data["personalised"]:
    tickers = [h["ticker"] for h in portfolio_data["holdings"]]
    print(f"Portfolio loaded: {', '.join(tickers[:8])}"
          + (f" ... (+{len(tickers)-8} more)" if len(tickers) > 8 else ""))
else:
    print("Running in GENERIC mode — run Notebook 12 for personalised education.")
print()
print("Try: 'What is alpha?' / 'How do interest rates affect my bonds?'")
print("     'Explain the expense ratio of my ETFs' / 'What is rebalancing?'")
print("Type 'quit' to stop.")
print(DIVIDER)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nSession ended.")
        break
    response = chat(user_input, session_id="interactive")
    print(f"\nAgent: {response}")

## Summary

### What Changed from the Previous Version

| Feature | Before (hardcoded) | Now (dynamic) |
|---------|-------------------|---------------|
| Portfolio source | Tickers hard-coded in prompt string | Loaded from `consolidated_portfolio.json` |
| Updates automatically | No — must edit code | Yes — re-run reload cell or restart notebook |
| Dollar values in context | Not available | Included when NB12 computed them |
| Source file tracking | None | Shows which files contributed each holding |
| No NB12 file present | Used hardcoded data silently | Warns user, falls back to generic mode |
| NB11 portfolio.json | Not supported | Also accepted (auto-detected) |

### Architecture
```
consolidated_portfolio.json   ← written by Notebook 12
         │
         ▼
 load_portfolio_context()     ← reads holdings, builds context block
         │
         ▼
 build_system_prompt()        ← embeds live holdings into system prompt
         │
         ▼
 assistant (LangGraph node)   ← injects SYSTEM_PROMPT on every turn
         │
  tools? ──► search_web       ← current prices / news for holdings
         │
        END
```

### 4-Step Teaching Framework
Every concept explanation follows:
1. **Simple Definition** — one sentence, no jargon
2. **Real-World Analogy** — non-financial, intuitive comparison
3. **Portfolio Application** — references the investor's real tickers
4. **Practical Takeaway** — one concrete consideration to act on

### Topics Covered
| Topic | Examples |
|-------|----------|
| Financial Terms | Beta, Alpha, Sharpe Ratio, P/E, Duration, Yield |
| Asset Allocation | Growth vs income, MPT, correlation |
| Fund Structure | ETF vs Mutual Fund, expense ratios, trading |
| Macro Impact | Interest rates, inflation, sector rotation |
| Tax Efficiency | Roth vs IRA, tax-loss harvesting, asset location |
| Risk Metrics | Volatility, max drawdown, concentration risk |

### Three-Agent System Flow
```
Notebook 11 (Designer)
    └── portfolio.json
            │
Notebook 12 (Consolidator & Analyst)
    └── consolidated_portfolio.json
            │
Notebook 13 (Education Agent)   ← this notebook
    └── Personalised chat responses
```